병합 llm 정의 완료


In [38]:
# app/services/llm_service.py
from typing import List, Dict, Any, Optional
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_postgres import PGVector
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA
import json
import re
import asyncio


class LLMService:
    def __init__(self):
        """langchain_ollama 기반 LLM 서비스 초기화"""

        # if settings.LLM_PROVIDER == "openai":
        #     # OpenAI LLM
        #     self.llm = ChatOpenAI(
        #         model=settings.OPENAI_MODEL,
        #         temperature=0.7,
        #         api_key=settings.OPENAI_API_KEY,
        #         max_tokens=2000,
        #     )

        #     # Ollama Embeddings
        #     self.embeddings = OllamaEmbeddings(
        #         model="nomic-embed-text",
        #         base_url=settings.OLLAMA_BASE_URL,
        #     )

        #     print("OpenAI LLM 및 Embeddings 초기화 완료")

        # else:  # ollama (기본값)
            # Ollama LLM
        self.llm = OllamaLLM(
            # model=settings.OLLAMA_MODEL,
            model="llama3.1:8b",
            temperature=0.2,  # 더 결정적인 출력을 위해 낮춤
            # base_url=settings.OLLAMA_BASE_URL,
            num_predict=512,  # 토큰 수 증가
            model_kwargs={
                "repeat_penalty": 1.1,
                "top_k": 40,
                "top_p": 0.9,
            },
        )

        # Ollama Embeddings
        self.embeddings = OllamaEmbeddings(
            model="nomic-embed-text",
            # base_url=settings.OLLAMA_BASE_URL,
        )

        print("Ollama LLM 및 Embeddings 초기화 완료")

        # Text Splitter (공통)
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len,
            separators=[
                "\n\n",   # 문단
                "\n",     # 줄
                "。", ".", "!", "?",  # 문장 끝
                # ":", "：", "-", "–",
                # " ",      # 띄어쓰기
                # ",", "，", "·",
            ]
        )

    def split_text(self, text: str) -> List[Document]:
        # 2) 텍스트 → LangChain Document 로 변환
        docs = [
            Document(page_content=t, metadata={"id": idx})
            for idx, t in enumerate(text)
        ]
        split_docs = self.text_splitter.split_documents(docs)

        return split_docs

        # """텍스트를 청크로 분할"""
        # chunks = self.text_splitter.split_text(text)

        # documents = []
        # for i, chunk in enumerate(chunks):
        #     doc = Document(
        #         page_content=chunk,
        #         metadata={
        #             **metadata,
        #             "chunk_index": i,
        #             "total_chunks": len(chunks)
        #         }
        #     )
        #     documents.append(doc)

        # return documents

    async def generate_summary(self, text: str) -> str:
        """이력서 요약 생성 - 강화된 프롬프트"""
        prompt = PromptTemplate(
            template="""# 역할
당신은 20년 경력의 HR 전문가이자 이력서 분석 전문가입니다.

# 작업
아래 이력서를 **3-4문장**으로 정확하고 간결하게 요약하세요.

# 분석 기준
1. **핵심 경력**: 주요 직무와 총 경력 기간
2. **기술 역량**: 보유한 핵심 기술 스택 (3-5개)
3. **주요 성과**: 정량적 성과나 대표 프로젝트
4. **차별점**: 이 지원자만의 강점

# 요약 형식
- 첫 문장: [이름]은(는) [총 경력]의 [주요 직무] 전문가입니다.
- 둘째 문장: [핵심 기술 스택]에 능숙하며...
- 셋째 문장: [주요 프로젝트/성과]를 달성했습니다.
- 넷째 문장: [차별화된 강점]을 보유하고 있습니다.

# 이력서 내용
{text}

# 요약 (한국어, 3-4문장, 명확하고 간결하게)
""",
            input_variables=["text"]
        )

        chain = prompt | self.llm | StrOutputParser()

        try:
            summary = await chain.ainvoke({"text": text[:4000]})

            # 불필요한 프리픽스 제거
            summary = summary.strip()
            summary = re.sub(r'^(요약:|Summary:)\s*', '', summary, flags=re.IGNORECASE)

            return summary
        except Exception as e:
            print(f"요약 생성 실패: {e}")
            return "이력서 요약 생성 중 오류가 발생했습니다."

    def analyze_strengths(self, vectordb: str) -> str:
        """강점 분석 - 강화된 프롬프트"""
        
        template="""# 역할
당신은 기술 인재 평가 전문가입니다. 20년간 수천 명의 개발자를 평가해왔습니다.

# 작업
이력서를 분석하여 **3가지 핵심 강점**을 찾아내세요.

# 분석 기준
각 강점은 반드시 다음을 포함해야 합니다:
1. **구체적 근거**: 이력서에서 확인 가능한 사실
2. **정량적 지표**: 경력 기간, 프로젝트 수, 성과 등
3. **차별화 요소**: 다른 지원자와 구분되는 점

# 출력 형식
각 강점을 다음 형식으로 작성하세요:
• [강점 제목]: [구체적 설명 2-3줄, 근거 포함]

# 예시
• 풀스택 개발 역량: 5년간 React, Node.js, PostgreSQL을 활용한 10개 이상의 프로젝트를 성공적으로 완수했으며, 프론트엔드부터 백엔드, 데이터베이스까지 전 영역을 독립적으로 처리할 수 있습니다.

# 3가지 핵심 강점 (한국어, 각 2-3줄, 구체적 근거 포함)
"""


        return self.no_use_rag(vectordb, template)

    async def generate_interview_questions(
            self,
            text: str,
            position: str
    ) -> List[Dict[str, str]]:
        """면접 질문 생성 - 강화된 프롬프트"""
        prompt = PromptTemplate(
            template="""# 역할
당신은 {position} 포지션의 면접관입니다. 지원자의 역량을 정확히 평가할 수 있는 질문을 만들어야 합니다.

# 작업
아래 이력서를 기반으로 **5개의 면접 질문**을 생성하세요.

# 질문 요구사항
1. **기술 질문 (2개)**: 이력서에 언급된 구체적 기술이나 프로젝트 관련
2. **경험 질문 (2개)**: 실제 경험과 문제 해결 능력 확인
3. **행동 질문 (1개)**: 협업, 리더십, 태도 관련

# 질문 난이도
- easy: 기본 지식이나 경험 확인
- medium: 실무 적용 능력 평가
- hard: 깊이 있는 이해도와 문제 해결 능력

# 이력서 내용
{text}

# 출력 형식
반드시 **유효한 JSON 배열만** 출력하세요. 다른 텍스트는 절대 포함하지 마세요.

[
    {{"question": "구체적인 질문 (한국어)", "category": "technical", "difficulty": "medium"}},
    {{"question": "구체적인 질문 (한국어)", "category": "technical", "difficulty": "hard"}},
    {{"question": "구체적인 질문 (한국어)", "category": "experience", "difficulty": "medium"}},
    {{"question": "구체적인 질문 (한국어)", "category": "experience", "difficulty": "hard"}},
    {{"question": "구체적인 질문 (한국어)", "category": "behavioral", "difficulty": "medium"}}
]

# 예시 (참고용)
[
    {{"question": "이력서에 언급된 FastAPI 프로젝트에서 가장 어려웠던 성능 최적화 경험을 구체적으로 설명해주세요.", "category": "technical", "difficulty": "hard"}},
    {{"question": "PostgreSQL과 Redis를 함께 사용하셨다고 했는데, 각각 어떤 용도로 활용하셨나요?", "category": "technical", "difficulty": "medium"}},
    {{"question": "프로젝트 일정이 촉박한 상황에서 예상치 못한 기술적 문제가 발생했을 때 어떻게 대처하셨나요?", "category": "experience", "difficulty": "medium"}},
    {{"question": "주니어 개발자와 함께 작업할 때 코드 리뷰를 어떻게 진행하셨나요? 구체적인 예시를 들어주세요.", "category": "experience", "difficulty": "hard"}},
    {{"question": "팀원과 기술적 의견 충돌이 있었던 경험과 해결 과정을 말씀해주세요.", "category": "behavioral", "difficulty": "medium"}}
]

# JSON 출력 (다른 텍스트 없이 JSON만)
""",
            input_variables=["text", "position"]
        )

        chain = prompt | self.llm | StrOutputParser()

        try:
            result = await chain.ainvoke({"text": text[:4000], "position": position})

            # JSON 추출 (마크다운 코드블록 제거)
            result = result.strip()
            result = re.sub(r'```json\s*', '', result)
            result = re.sub(r'```\s*', '', result)

            # JSON 파싱
            json_match = re.search(r'\[.*\]', result, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                questions = json.loads(json_str)

                # 유효성 검증
                if len(questions) >= 5:
                    return questions[:5]
                else:
                    return self._get_default_questions(position)
            else:
                return self._get_default_questions(position)

        except Exception as e:
            print(f"면접 질문 생성 실패: {e}")
            return self._get_default_questions(position)

    def _get_default_questions(self, position: str) -> List[Dict[str, str]]:
        """기본 면접 질문 - 개선된 버전"""
        return [
            {
                "question": f"{position} 포지션에 지원하신 이유와 이 직무에서 이루고 싶은 목표를 구체적으로 말씀해주세요.",
                "category": "behavioral",
                "difficulty": "easy"
            },
            {
                "question": "가장 기술적으로 도전적이었던 프로젝트 경험을 선택해서, 문제 상황, 해결 과정, 결과를 단계별로 설명해주세요.",
                "category": "experience",
                "difficulty": "hard"
            },
            {
                "question": "최근 학습한 기술이나 도구가 있다면, 왜 배우게 되었고 어떻게 활용하고 있는지 말씀해주세요.",
                "category": "technical",
                "difficulty": "medium"
            },
            {
                "question": "프로젝트 데드라인이 촉박한 상황에서 예상치 못한 버그가 발생했을 때, 어떤 우선순위로 문제를 해결하시나요?",
                "category": "experience",
                "difficulty": "medium"
            },
            {
                "question": "동료와 기술적 의견이 충돌했던 경험이 있다면, 어떻게 합의점을 찾으셨는지 구체적으로 설명해주세요.",
                "category": "behavioral",
                "difficulty": "medium"
            }
        ]

    async def extract_skills(self, text: str) -> Dict[str, int]:
        """스킬 추출 및 점수화 - 강화된 프롬프트"""
        prompt = PromptTemplate(
            template="""# 역할
당신은 기술 스택 평가 전문가입니다. 이력서에서 기술 역량을 정확히 분석해야 합니다.

# 작업
아래 이력서에서 **기술 스킬**을 추출하고 숙련도를 평가하세요.

# 분석 기준
각 기술의 점수(0-100)는 다음을 고려하여 책정:
1. **사용 기간**: 언급된 경력 기간
2. **프로젝트 수**: 해당 기술을 사용한 프로젝트 개수
3. **깊이**: 단순 사용 vs 심화 활용 (최적화, 아키텍처 설계 등)
4. **최신성**: 최근에도 사용 중인지

# 점수 가이드
- 90-100점: 5년 이상, 전문가 수준, 다수 프로젝트, 최신 활용
- 80-89점: 3-5년, 능숙, 여러 프로젝트
- 70-79점: 2-3년, 실무 가능
- 60-69점: 1-2년, 기본 활용
- 50-59점: 1년 미만, 학습 중

# 이력서 내용
{text}

# 출력 형식
반드시 **유효한 JSON 객체만** 출력하세요. 다른 텍스트는 절대 포함하지 마세요.

{{"Python": 90, "FastAPI": 85, "PostgreSQL": 80, "Docker": 75, "AWS": 70, "React": 65}}

# 주의사항
- 프로그래밍 언어, 프레임워크, 데이터베이스, 도구 등만 포함
- 소프트 스킬(커뮤니케이션 등)은 제외
- 이력서에 명시된 기술만 포함 (추측 금지)
- 최소 3개, 최대 15개 기술
- 점수는 반드시 정수 (0-100)

# JSON 출력 (다른 텍스트 없이 JSON만)
""",
            input_variables=["text"]
        )

        chain = prompt | self.llm | StrOutputParser()

        try:
            result = await chain.ainvoke({"text": text[:4000]})

            # JSON 추출
            result = result.strip()
            result = re.sub(r'```json\s*', '', result)
            result = re.sub(r'```\s*', '', result)

            json_match = re.search(r'\{.*\}', result, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                skills = json.loads(json_str)

                # 유효성 검증 (점수가 0-100 범위인지)
                validated_skills = {
                    k: min(100, max(0, v))
                    for k, v in skills.items()
                    if isinstance(v, (int, float))
                }

                return validated_skills
            else:
                return {}

        except Exception as e:
            print(f"스킬 추출 실패: {e}")
            return {}

    def extract_work_experience(self, vectordb: str) -> List[Dict[str, Any]]:
        template="""# 역할
당신은 경력 사항 분석 전문가입니다. 이력서에서 직무 경력을 정확히 추출해야 합니다.

# 작업
아래 이력서에서 **직무 경력**을 시간순으로 추출하세요.

# 추출 정보
각 경력마다 다음 정보를 포함:
1. **직책/포지션**: 정확한 직책명
2. **회사명**: 근무한 회사 또는 조직
3. **근무 기간**: "YYYY.MM - YYYY.MM" 또는 "YYYY.MM - Present" 형식
4. **주요 성과**: 구체적인 성과나 프로젝트 (2-5개)

# 출력 형식
반드시 **유효한 JSON 배열만** 출력하세요. 다른 텍스트는 절대 포함하지 마세요.

[
    {{
        "title": "Senior Backend Engineer",
        "company": "테크컴퍼니",
        "period": "2022.03 - Present",
        "achievements": [
            "MSA 아키텍처 기반 주문 시스템 설계 및 구축 (일 10만 건 처리)",
            "FastAPI + PostgreSQL 조합으로 API 응답 속도 70% 개선",
            "Docker + Kubernetes 기반 CI/CD 파이프라인 구축"
        ]
    }},
    {{
        "title": "Backend Developer",
        "company": "스타트업",
        "period": "2020.01 - 2022.02",
        "achievements": [
            "Django 기반 관리자 시스템 개발 및 유지보수",
            "RESTful API 20개 이상 설계 및 구현",
            "MySQL 데이터베이스 쿼리 최적화로 조회 속도 50% 향상"
        ]
    }}
]

# 주의사항
- 최신 경력부터 시간순으로 나열
- 인턴, 프리랜서, 정규직 모두 포함
- 성과는 구체적이고 정량적으로 작성
- 이력서에 없는 정보는 추측하지 말 것
- 회사경력이 아닌 프로젝트 경력은 강력하게 무조건 제외할 것

# JSON 출력 (다른 텍스트 없이 JSON만)
"""
        result = self.no_use_rag(vectordb, template)

        try:
            # JSON 추출
            result = result.strip()
            result = re.sub(r'```json\s*', '', result)
            result = re.sub(r'```\s*', '', result)

            json_match = re.search(r'\[.*\]', result, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                experiences = json.loads(json_str)
                return experiences
            else:
                return []

        except Exception as e:
            print(f"경력 추출 실패: {e}")
            return []

    def get_vectorstore(
            self,
            split_docs
            # connection_string: str,
            # collection_name: str = "resume_vectors"
    ) -> Chroma:
        # """PGVector 초기화 (langchain_postgres 사용)"""
        # vectorstore = PGVector(
        #     embeddings=self.embeddings,
        #     collection_name=collection_name,
        #     connection=connection_string,
        #     use_jsonb=True,
        # )
        
        embeddings = OllamaEmbeddings(model="nomic-embed-text")
        
        vectorstore = Chroma.from_documents(
            documents=split_docs,
            embedding=self.embeddings,
            persist_directory=None,  # 디스크 저장 안 하고 메모리만 쓰고 싶으면 None
        )
        
        return vectorstore

    async def rag_query(
            self,
            vectorstore: PGVector,
            question: str,
            doc_id: str,
            k: int = 5
    ) -> Dict[str, Any]:
        """RAG 기반 질의응답 - 강화된 프롬프트"""

        try:
            # 유사도 검색
            retriever = vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={
                    "k": k,
                    "filter": {"doc_id": doc_id}
                }
            )

            # 관련 문서 검색
            docs = await retriever.ainvoke(question)

            if not docs:
                return {
                    "answer": "해당 이력서에서 관련된 정보를 찾을 수 없습니다.",
                    "chunks": [],
                    "relevance_score": 0.0
                }

            # 컨텍스트 구성
            context = "\n\n".join([doc.page_content for doc in docs])

            # RAG 프롬프트 - 강화 버전
            prompt = ChatPromptTemplate.from_template(
                """# 역할
당신은 이력서 분석 전문가입니다. 주어진 이력서 정보를 바탕으로 정확하게 답변해야 합니다.

# 지침
1. **사실 기반 답변**: 아래 이력서 내용에만 근거하여 답변
2. **근거 명시**: 답변의 근거가 되는 부분을 언급
3. **정직한 답변**: 정보가 없으면 "이력서에 해당 정보가 없습니다"라고 답변
4. **구체적 답변**: 모호하지 않고 명확하게 답변
5. **한국어 답변**: 자연스러운 한국어로 작성

# 이력서 내용 (참고용)
{context}

# 질문
{question}

# 답변 (한국어, 2-4문장, 근거 포함)
""")

            chain = prompt | self.llm | StrOutputParser()
            answer = await chain.ainvoke({
                "context": context,
                "question": question
            })

            return {
                "answer": answer.strip(),
                "chunks": [doc.page_content for doc in docs],
                "relevance_score": 0.95
            }

        except Exception as e:
            print(f"RAG 쿼리 실패: {e}")
            return {
                "answer": f"답변 생성 중 오류가 발생했습니다: {str(e)}",
                "chunks": [],
                "relevance_score": 0.0
            }


    
    def use_rag(self, vectordb: str, template: str, k: int) -> str:
        retriever = vectordb.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k},   # 관련 텍스트 3개 정도만 사용
        )
        
        # 5) RAG + 요약 체인 (RetrievalQA 사용 – query를 요약 지시로 주면 됨)
        qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            retriever=retriever,
            chain_type="stuff",         # 검색된 텍스트를 한 번에 LLM에 넣는 방식
            return_source_documents=True,
        )

        result = qa_chain.invoke({"query": template})
        answer = result["result"]
        sources = result["source_documents"]
    
        print("================================== 질문 ===============================")
        print(template)
        print("\n============================== 요약 결과 ============================")
        print(answer)
        print("\n========================= 사용된 텍스트 조각 ========================")
        for i, d in enumerate(sources):
            print(f"[{i}] {d.page_content}")

        return answer
        
    def no_use_rag(self, vectordb: str, template: str) -> str:
        template += f"""

# 이력서 내용
{vectordb}
"""
        
        prompt = PromptTemplate(
            template=template,
            input_variables=["text"]
        )

        chain = prompt | self.llm | StrOutputParser()

        try:
            answer = chain.invoke({"text": text[:4000]})
            answer = answer.strip()

            print("================================== 질문 ===============================")
            print(template)
            print("\n============================== 요약 결과 ============================")
            print(answer)

            return answer
        except Exception as e:
            print(f"생성 실패: {e}")
            return "이력서 요약 생성 중 오류가 발생했습니다."

    
    def should_merge(self, prev_text: str, current_text: str) -> bool:
        """
        prev_text : 바로 이전 조각
        current_text : 새로 들어온 조각
        """
        prompt = f"""
너는 이력서 텍스트 조각들을 묶는 도우미야.

[규칙]
- 두 텍스트가 같은 항목/섹션에 속한다고 느껴질 때만 "YES"를 출력해.
  예시:
    - 같은 회사 경력의 직무 설명
    - 같은 프로젝트의 상세 설명 여러 줄
    - 같은 학교/학력에 대한 여러 줄 설명
- 회사/학교/연도/직무/섹션 제목 등이 바뀌면 "NO"를 출력해.
- 애매하면 무조건 "NO"로 판단해.
- "YES" 또는 "NO" 한 글자만 출력해.

[바로 이전 텍스트]
{prev_text}

[새로 들어온 텍스트]
{current_text}

답:
""".strip()

        res = self.llm.invoke(prompt)
        answer = res.strip().upper()
        return answer.startswith("Y")



    
    # def group_texts_with_llm(self, texts: List[str]) -> List[str]:
    #     """
    #     texts: 이력서에서 뽑은 줄/조각 리스트
    #     return: LLM이 의미 단위로 묶어준 큰 덩어리 리스트 (str 리스트)
    #     """
    #     if not texts:
    #         return []
    
    #     groups: List[str] = []
    #     current_group: List[str] = [texts[0]]
    
    #     for t in texts[1:]:
    #         prev_text = current_group[-1]  # ✅ 바로 전 한 줄만 비교
    
    #         if self.should_merge(prev_text, t):
    #             # 같은 섹션 → 같은 그룹에 붙이기
    #             current_group.append(t)
    #         else:
    #             # 다른 섹션 → 기존 그룹 확정, 새 그룹 시작
    #             groups.append("\n\n".join(current_group))
    #             current_group = [t]
    
    #     # 마지막 그룹 추가
    #     groups.append("\n\n".join(current_group))
    #     return groups



# 싱글톤 인스턴스
llm_service = LLMService()

print("llmService 인스턴스화 완료")


Ollama LLM 및 Embeddings 초기화 완료
llmService 인스턴스화 완료


In [8]:
# ============================================
# 1. 텍스트 추출
# ============================================
print(f"[Step 1] 텍스트 추출 중...")

texts = ['지원분야 : 프론트&백엔드 개발자(신입/경력) 모집', '입사지원일 : 2024년 03월 05일 (화)', '이영선 신입', '여, 1996 (27세)', 'dudtjs960630@naver.com 010-3325-2804', '(07368) 서울 영등포구 도신로29가길', '학력 대학교(2,3년) 졸업 신구대학교 전공 치위생과 업무경험 - 희망연봉 회사내규에 따름 포트폴리오 https://www.…', '나의 스킬', 'css HTML SQL Eclipse jQuery Ajax Maven JavaScript Spring Maria DB MyBatis JSP', 'Java', '학력', '대학교(2,3년) 졸업', '신구대학교(2·3년제) (주간)', '2015.03 ~ 2018.03 (졸업)', '치위생과', '지역 경기 논문/작품 수도권 주민들의 스마트폰 사용이 우울감에 미치는 영향', '심석고등학교', '문과계열', '2012.03 ~ 2015.02 (졸업)', '경험/활동/교육', '구디아카데미', '공공데이터 융합 자바/스프링 개발자 양성과정(2) 총 113일/900시간', 'WEB Front-End : Html,CSS,Java Script,JQuery,JSP -Data Base : mariaDB -Web Back-End : JAVA,Spring Framework,Mybatis', '치아나라234', '업무형태 : 정규직 부서 및 직급 : 진료실 역할 : 통합 진료 팀 팀원,임플란트 팀 팀원. 스케일링 센터 담당 퇴직사유: 직무변경', '2023.01 ~ 2023.07', '2019.04 ~ 2022.04', '서현이바른치과', '2018.03 ~ 2019.03', '업무형태 : 정규직 부서 및 직급 : 진료실 역할 : 진료실 퇴직사유:이직희망', '자격/어학/수상', '정보처리산업기사 (필기합격) 2023.06 한국산업인력공단 치과위생사 (최종합격) 2018.02 한국보건의료인국가시험원 글로벌챌린지 2017.01 신구대학교 EFG ucc 챌린지 2016.07 efg', '포트폴리오 및 기타문서', '기타', 'https://velog.io/@yslys', '포트폴리오', 'https://www.notion.so/eda47922f74748e4943f5a81d983f561?pvs=4', '* 첨부파일 다운로드는 사람인 기업회원만 가능하며 개인간 공개는 제한됩니다.', '자기소개서', '기술 역량 프로젝트 활동', '[1차 프로젝트] [개인프로젝트] 1.구현언어: javaScript, HTML', '개발 내용 : 카드 뒤집기 게임 구현기능 : 카드의 짝을 맞추는 게임', '[2차 프로젝트] 개발환경 및 라이브러리 1. 구현언어 : Java, HTML, JavaScript, jQuery, SQL', '2. WAS : Tomcat 9.0 3. DB 서버 : mysql, mariadb 4. 형상관리 : git 5.framework :sitemesh', '기간:2023.04.17-2023.05.12 인원: 2명 개발내용: 스포츠 구단 사이트 담당업무 : 회원관리(로그인시 정규화),(게시판 관리, 메인 스와이프기능, 조회성 화면(오시는길,응원가,경기일 정,경기기록)', '확인 구현.', '-회원가입 페이지 : 아이디 중복검사, 비밀번호 일치여부 -아이디,패스워드 찾기 -로그인.로그아웃 -회원정보수정 -게시판 글쓰기,수정,삭제 -관리자만 공지사항 글쓰기 -선수단만 선수 게시판 글쓰기 -댓글기능(삭제,수정) -시즌에 따른 경기 일정, 경기 기록 확인 페이지 -메인 화면에 선수단 스와이프 기능', '[Spring 기반 3차 프로젝트]', '개발환경 및 라이브러리 1. 구현언어 : Java,HTML, JavaScript, jQuery, SQL 2. WAS : Tomcat 9.0 3. DB 서버 : mysql, mariadb 4. 형상관리 : git 5.framework :springframework,sitemesh', '개발 내용-식당 예약 사이트(밥티켓) 기간:2023.06.12-2023.07.12 인원: 2명 담담업무 : 가게 등록 ,일반 회원 예약 등록 및 관리, 사장 회원 가게 수정, 삭제, 예약 페이지, 가게 상세보기 페 이지, 사장회원 예약 관리(승인 ,거절), 예약 상태 확인 -일반 회원 예약 관리(예약 취소), 예약 상태 확인, 이용 완료 시 별점 등록', '가게 등록', ':', '- 가게 등록 시 휴무일 선택 가능 - 가게 등록 시 open, close 시간 입력 받아 가게 마다 예약 가능 시간 상이함 - 가게 등록 시 주소는 도로명주소 api 사용 - 메뉴 테이블과 조인하여 사용 - 가게 이미지 등록 가능, 미등록 시 기본 이미지 등록 됨', '가게 리스트', '- 비회원도 가게 리스트 조회 가능 -가게 리스트에서 평균 별점 확인 가능 -가게 리스트에서 위치, 메뉴, 가게 이름으로 검색 가능', '가게 상세 페이지 - 카카오 지도를 통해 가게 위치 파악 가능 -예약 버튼은 로그인한 사람에게만 보여짐', '예약 페이지 -가게 사장이 등록 한 휴무일에 따라 달력에 예약 가능 날짜가 표시됨 -가게 사장이 등록한 open, close 시간에 따라 예약 가능 시간이 표시됨', '-가게 사장이 등록한 최대 인원수에 따라 예약 인원이 다 찰 경우 예약 인원 마감 및 예약 가능 인원 alert 창 을 통해 안내함', '- 카카오 페이 결제 api를 사용하여 예약금 결제', '예약 목록(회원)', '- 내 예약 상태를 수정,확인 가능함(승인,거절,대기,이용완료) -예약 상세 보기 및 수정 페이지에서 예약자 이름, 전화번호 수정 가능 - 대기 상태에선 취소 가능(취소 시 예약자에게 취소 문자 발송) -예약 날짜가 지나 이용 완료 상태로 변경 시 별점 등록 가능한 버튼이 보여짐 -별점 등록 시 내가 등록 한 별점 확인 가능', '내 가게 목록(사장)', '- 내가 등록 한 가게 목록 확인 가능함 -가게 마다 등록 된 예약 확인 가능함 - 예약 상세 페이지에서 예약 승인,거절 가능(승인,거절 시 예약자에게 문자 발송) 취소 이용완료 등 이용 상태 확인 가능', '- 예약 승인이 한 타임당 최대 인원수를 채우면 예약 페이지에서 예약 불가능 안내창 띄움 - 가게 수정 가능 - 가게 삭제 가능', '-가게 삭제 전 안내창 띄우며 가게 삭제 시 컬럼에 update 하여 삭제 표시, 삭제 시 가게 리스트, 내 가게 목록 에선 조회 불가능', '[프로젝트 중 어려웠던 부분과 해결책]', 'Ajax를 사용하는 과정에서 데이터는 정상적으로 저장이 되는데 script 부분에서는 에러가 자꾸 발생해서 힘들 었습니다. 찾아보니 controller 부분에서 @responsebody를 사용하지 않아 에러가 발생했던 거여서 @responsebody를 작성해서 해결했습니다.', '최종 프로젝트에서 예약과 가게 부분을 맡았는데 예약 상태 부분이나, 메뉴 휴무일 등에서 생각보다 세세하 게 구현해야 되는 기능들이 많았습니다. 처음에는 막막했지만 차근차근 계획을 세우고 실행하는게 중요하다 고 생각해 구글 시트를 통해 구현해야 되는 부분을 크게 나누고 그 안에서 세세하게 기능들을 정리하여 날마 다 체크하는 방식으로 진행하다 보니 원했던 기능들을 기간안에 구현 할 수 있었습니다.', '지원동기', '2018년부터 약 4년간 치과위생사로 근무하며 느낀 불편함이 있었습니다. 그건 바로 진료실에서 사용하는 ‘전자차트’였습니다. 환자가 집중된 시간엔 빠르게 차트를 보고 환자를 응대하고 작성 마무리를 지어야 하는 과정에서 한눈에 전 체적으로 들어오지 않아 차팅 시 시간이 길어지거나 하는 불편사항들이 있었습니다. 업무를 하며 검색 기능을 추가하거나 조금 더 빠르고 간편하게 이전 진료 기록을 찾아볼 수 있었으면 좋겠다 는 생각을 하다가 이런 전자차트는 어떻게 만들어지는지, 환자의 데이터는 어떻게 저장이 되고 사용할 수 있 는지에 대한 호기심이 생겼습니다. 개발을 통해서 만들 수 있다는 것을 알게 되고, 관심을 갖고 주위를 둘러보니 차트 프로그램 말고도 실생활에 서 사용하는 다양한 프로그램들이 있었고, 이러한 것들을 직접 만들어보고 싶다는 생각을 하게 되었습니다. 공부를 하는 지금 낯설고 생소한 기술들이 다양하지만 배움을 통해 제가 생각했던 모습들에 한 발자국 더 다 가가 다양한 프로그램을 구현하는 개발자가 되고 싶었습니다.', '성격의 장단점', '소통, 센스 꼼꼼함이 제 성격의 핵심 키워드라 생각합니다.', '소통, 센스, 꼼꼼함', '제가 일했던 직무 특성상 환자와의 소통,팀원과의 소통, 원장님과의 소통이 잘 연결이 되어야 원활한 진료를 진행 할 수 있었기 때문에 소통과 센스는 굉장히 중요하면서도 기본적인 부분이었습니다.', '일을 하다 보면 응대가 어려운 환자들도 많이 내원 하는데 환자가 본인의 요구를 명확히 표현하지 못해도 환 자의 요구를 잘 파악하고 응대하는 것을 잘했기 때문에 환자들께서 칭찬도 많이 해주시고 제 이름을 외워 다 음 번 내원 시 진료 담당을 원하시는 경우도 종종 있었으며 그 모습을 장점으로 봐주신 상사 덕분에 진료실 업무 뿐만 아니라 상담 업무도 맡게 되었습니다. 또한 직업 특성상 꼼꼼함이 중요했습니다. 진료 중이나 진료 후 원장님과 환자와의 대화나 환자의 치료 계획 을 빠짐없이 기록을 남기는 것이 좋기 때문에 항상 모든 내용을 빠짐없이 기록해서 작성 된 차트만 보고도 그 진료실의 상황을 알 수 있도록 했습니다. 매일 환자 차트 리뷰를 하다 보니 작성 된 문서를 잘 읽고 파악할 수 있는 능력도 자연스럽게 향상 되었습니 다.', '소심함', '저는 생각을 많이 하고 소심한 경향이 있습니다. 하지만 이렇게 혼자 생각을 많이 하는 성격이 개발자의 직업 으로 좋은 부분은 아니라고 생각이 들었습니다. 그 후 학원에서 수업을 하거나 프로젝트를 진행할 때 잘 안 풀리는 부분이 있으면 3번까지는 스스로 생각해 해결해 보려고 하고 도움이 필요할 땐 도움을 요청할 수 있 는 성격으로 바꾸고 있습니다.', '위의 모든 내용은 사실과 다름 없음을 확인합니다. 입사지원일 : 2024년 03월 05일 (화)/ 작성자 : 이영선', '위조된 문서를 등록하여 취업활동에 이용시 법적 책임을 지게 될 수 있습니다. (주)사람인은 구직자가 등록한 문서에 대해 보증하거나 별도의 책임을 지지 않으며 첨부된 문서를 신뢰하여 발생한 법적 분쟁에 책임을 지지 않습니다. (개인회원약관 제16조 관련) 또한 구인/구직 목적 외 목적으로 이용 시 이력서 삭제 혹은 비공개 조치될 수 있습니다.']

# text = llm_service.group_texts_with_llm(texts)

print(f"[Step 1] 추출 완료\n{len(texts)}")
# for i, doc in enumerate(text):
#     print(f"  - content {i}: {doc}")

[Step 1] 텍스트 추출 중...
[Step 1] 추출 완료
79


In [9]:
# ---------------- 병합 ---------------- #

# 1) 병합용 LLM 인스턴스 준비 (Ollama 예시)
merge_llm = ChatOllama(
    model="qwen2.5:7b-instruct",  # 또는 llama3.1, solar 등
    temperature=0.0,
)

grouper = ResumeBlockGrouper(merge_llm, debug=True)

blocks = grouper.group_lines(texts)

print("\n===== 최종 병합 블록 =====")
for i, b in enumerate(blocks):
    print(f"\n=== 블록 {i} ===")
    print(b)

===== [DEBUG] Prompt =====
너에게 이력서에서 추출한 텍스트 줄들이 있다. 각 줄에는 인덱스가 붙어 있다.

목표:
- 의미가 같은 항목/섹션에 속하는 연속된 줄들을 하나의 그룹으로 묶어라.
- 예시 그룹:
  - 학력: 같은 학교/학과/재학·졸업 기간/학점/관련 활동 → 한 그룹
  - 경력: 같은 회사 경력(회사명, 직무, 재직 기간, 역할/성과 bullet들) → 한 그룹
  - 프로젝트: 같은 프로젝트(이름, 기간, 사용 기술, 역할, 결과) → 한 그룹

규칙:
- 인덱스는 반드시 오름차순이어야 한다.
- 각 그룹 안의 인덱스는 연속된 값이어야 한다.
  예: [0, 1, 2, 3], [4, 5], [6], [7, 8, 9]
- 어떤 줄도 빠뜨리지 말고, 모든 인덱스가 정확히 한 번씩만 등장해야 한다.

출력 형식:
- 반드시 JSON 배열만 출력하라.
- 형식 예시:
[
  [0, 1, 2, 3],
  [4, 5],
  [6]
]
- JSON 외의 다른 텍스트, 설명, 주석은 절대 출력하지 마라.

[텍스트 목록]
0: 지원분야 : 프론트&백엔드 개발자(신입/경력) 모집
1: 입사지원일 : 2024년 03월 05일 (화)
2: 이영선 신입
3: 여, 1996 (27세)
4: dudtjs960630@naver.com 010-3325-2804
5: (07368) 서울 영등포구 도신로29가길
6: 학력 대학교(2,3년) 졸업 신구대학교 전공 치위생과 업무경험 - 희망연봉 회사내규에 따름 포트폴리오 https://www.…
7: 나의 스킬
8: css HTML SQL Eclipse jQuery Ajax Maven JavaScript Spring Maria DB MyBatis JSP
9: Java
10: 학력
11: 대학교(2,3년) 졸업
12: 신구대학교(2·3년제) (주간)
13: 2015.03 ~ 2018.03 (졸업)
14: 치위생과
15: 지역 경기 논문/작품 수도권 주민들의 스마트폰 사용이 우울감에 미치는 영향
16: 심석고등학

In [11]:


# ============================================
# 2. 텍스트 스플리팅
# ============================================
print(f"\n[Step 2] 텍스트 분할 중...")

documents = llm_service.split_text(
    text=blocks
)

print(f"[Step 2] 분할 완료: {len(documents)} chunks")
for i, doc in enumerate(documents[:]):
    print(f"  - Chunk {i}: {len(doc.page_content)} chars")
    print(f"--------------- content {i} --------------")
    print(f"{doc.page_content}\n")

# ============================================
# 3. 임베딩 & Vector DB 저장 ( 동기 방식)
# ============================================
# print(f"\n[Step 3] 임베딩 & Vector DB 저장 중...")



# for i, doc in enumerate(documents):
#     print(f"  - Embedding chunk {i + 1}/{len(documents)}...")

#     #  동기 방식으로 임베딩 생성
#     embedding = llm_service.embeddings.embed_query(doc.page_content)

#     # DB에 저장
#     vector_data = VectorStore(
#         doc_id=doc.metadata.get("doc_id"),
#         doc_name=doc.metadata.get("doc_name"),
#         chunk_index=doc.metadata.get("chunk_index"),
#         content=doc.page_content,
#         embedding=embedding,  # List[float]
#         meta_data=doc.metadata
#     )



# print(f"[Step 3] Vector DB 저장 완료: {len(text)} vectors")


[Step 2] 텍스트 분할 중...
[Step 2] 분할 완료: 16 chunks
  - Chunk 0: 61 chars
--------------- content 0 --------------
지원분야 : 프론트&백엔드 개발자(신입/경력) 모집
입사지원일 : 2024년 03월 05일 (화)
이영선 신입

  - Chunk 1: 74 chars
--------------- content 1 --------------
여, 1996 (27세)
dudtjs960630@naver.com 010-3325-2804
(07368) 서울 영등포구 도신로29가길

  - Chunk 2: 159 chars
--------------- content 2 --------------
학력 대학교(2,3년) 졸업 신구대학교 전공 치위생과 업무경험 - 희망연봉 회사내규에 따름 포트폴리오 https://www.…
나의 스킬
css HTML SQL Eclipse jQuery Ajax Maven JavaScript Spring Maria DB MyBatis JSP
Java

  - Chunk 3: 103 chars
--------------- content 3 --------------
학력
대학교(2,3년) 졸업
신구대학교(2·3년제) (주간)
2015.03 ~ 2018.03 (졸업)
치위생과
지역 경기 논문/작품 수도권 주민들의 스마트폰 사용이 우울감에 미치는 영향

  - Chunk 4: 34 chars
--------------- content 4 --------------
심석고등학교
문과계열
2012.03 ~ 2015.02 (졸업)

  - Chunk 5: 171 chars
--------------- content 5 --------------
경험/활동/교육
구디아카데미
공공데이터 융합 자바/스프링 개발자 양성과정(2) 총 113일/900시간
WEB Front-End : Html,CSS,Java Script,JQuery,JSP -Data Base : mariaDB -Web

In [12]:
# 3) 임베딩 + 벡터스토어(Chroma) + Retriever 생성

vector_data = llm_service.get_vectorstore(documents)
# llm_service = ""
# vector_data = ""

print("임베딩 완료")
before = len(vector_data.get()["ids"])
print("청크 수:", before)


In [30]:

print(f"\n[Step 4] 강점 분석 중...")

# strengths = llm_service.analyze_strengths(vector_data)
strengths = llm_service.analyze_strengths("\n\n".join(blocks))
if isinstance(strengths, Exception):
    print(f"  ⚠️ 강점 분석 실패: {strengths}")
    


[Step 4] 강점 분석 중...
================================== 질문 ===============================
# 역할
당신은 기술 인재 평가 전문가입니다. 20년간 수천 명의 개발자를 평가해왔습니다.

# 작업
이력서를 분석하여 **3가지 핵심 강점**을 찾아내세요.

# 분석 기준
각 강점은 반드시 다음을 포함해야 합니다:
1. **구체적 근거**: 이력서에서 확인 가능한 사실
2. **정량적 지표**: 경력 기간, 프로젝트 수, 성과 등
3. **차별화 요소**: 다른 지원자와 구분되는 점

# 출력 형식
각 강점을 다음 형식으로 작성하세요:
• [강점 제목]: [구체적 설명 2-3줄, 근거 포함]

# 예시
• 풀스택 개발 역량: 5년간 React, Node.js, PostgreSQL을 활용한 10개 이상의 프로젝트를 성공적으로 완수했으며, 프론트엔드부터 백엔드, 데이터베이스까지 전 영역을 독립적으로 처리할 수 있습니다.

# 3가지 핵심 강점 (한국어, 각 2-3줄, 구체적 근거 포함)


# 이력서 내용
지원분야 : 프론트&백엔드 개발자(신입/경력) 모집
입사지원일 : 2024년 03월 05일 (화)
이영선 신입

여, 1996 (27세)
dudtjs960630@naver.com 010-3325-2804
(07368) 서울 영등포구 도신로29가길

학력 대학교(2,3년) 졸업 신구대학교 전공 치위생과 업무경험 - 희망연봉 회사내규에 따름 포트폴리오 https://www.…
나의 스킬
css HTML SQL Eclipse jQuery Ajax Maven JavaScript Spring Maria DB MyBatis JSP
Java

학력
대학교(2,3년) 졸업
신구대학교(2·3년제) (주간)
2015.03 ~ 2018.03 (졸업)
치위생과
지역 경기 논문/작품 수도권 주민들의 스마트폰 사용이 우울감에 미치는 영향

심석고등학교
문과계열
2012.03 ~ 2015.02 (졸업)

경험/활동/교육
구디

In [39]:
print(f"\n[Step 4] 경력 분석 중...")

experience_task = llm_service.extract_work_experience(vector_data)
if isinstance(experience_task, Exception):
    print(f"  ⚠️ 경력 분석 실패: {experience_task}")


[Step 4] 경력 분석 중...
================================== 질문 ===============================
# 역할
당신은 경력 사항 분석 전문가입니다. 이력서에서 직무 경력을 정확히 추출해야 합니다.

# 작업
아래 이력서에서 **직무 경력**을 시간순으로 추출하세요.

# 추출 정보
각 경력마다 다음 정보를 포함:
1. **직책/포지션**: 정확한 직책명
2. **회사명**: 근무한 회사 또는 조직
3. **근무 기간**: "YYYY.MM - YYYY.MM" 또는 "YYYY.MM - Present" 형식
4. **주요 성과**: 구체적인 성과나 프로젝트 (2-5개)

# 출력 형식
반드시 **유효한 JSON 배열만** 출력하세요. 다른 텍스트는 절대 포함하지 마세요.

[
    {{
        "title": "Senior Backend Engineer",
        "company": "테크컴퍼니",
        "period": "2022.03 - Present",
        "achievements": [
            "MSA 아키텍처 기반 주문 시스템 설계 및 구축 (일 10만 건 처리)",
            "FastAPI + PostgreSQL 조합으로 API 응답 속도 70% 개선",
            "Docker + Kubernetes 기반 CI/CD 파이프라인 구축"
        ]
    }},
    {{
        "title": "Backend Developer",
        "company": "스타트업",
        "period": "2020.01 - 2022.02",
        "achievements": [
            "Django 기반 관리자 시스템 개발 및 유지보수",
            "RESTful API 20개 이상 설계 및 구현",
            "MySQL 데이터베이스 쿼리 최적화로 조회 